# WAYMO DATA MERGE: NHTSA Crash Reports + Waymo Safety Impact Data Hub

**Author:** Anders  
**Purpose:** Merge NHTSA crash data with Waymo's Safety Hub to create one enriched dataset

This script handles:
1. Loading the three source files (NHTSA prior, NHTSA post, Waymo Safety Hub CSV2)
2. Filtering for Waymo-only crashes in the NHTSA data
3. Harmonizing column names between Amendment 2 and Amendment 3 formats
4. Combining the two NHTSA files into one unified dataset
5. Deduplicating to keep only the latest version of each crash
6. Merging the combined NHTSA data onto the Waymo Safety Hub (left join)
7. Outputting the merged dataset + a separate file for "extra" NHTSA crashes not in the hub

**How to update with new data:** Just change the file paths in Step 1 and re-run all cells.

---

## Step 0: Import Libraries

In [4]:
import pandas as pd
import os

In [8]:
import os
print(os.getcwd())

/Users/AndersEidesvik/projects/bna-fuhgeddaboudit/merging-data-analysis-anders-NEW


## Step 1: Define File Paths

**When new data arrives, update ONLY these paths and re-run the entire notebook.**

The three input files:
- **NHTSA Prior:** Crash reports filed under Amendment 2 (before June 16, 2025)
- **NHTSA Post:** Crash reports filed under Amendment 3 (after June 16, 2025)
- **Waymo Safety Hub CSV2:** Waymo's curated crash data with their own classifications

In [7]:
# =============================================================================
# FILE PATHS — Update these when new data arrives, then re-run all cells
# =============================================================================

# NHTSA data files (raw crash reports from government)
NHTSA_PRIOR_PATH = "raw-data/nhtsa_ads_prior_june16_downloaded20250129.csv"
NHTSA_POST_PATH = "raw-data/nhtsa_ads_post_june16_downloaded_20250129.csv"

# Waymo's own curated data (Safety Impact Data Hub, CSV2)
WAYMO_HUB_PATH = "raw-data/waymo_safety_hub_csv2_downloaded_20250129.csv"

# Output files
OUTPUT_MERGED_PATH = "clean-data/waymo_merged_nhtsa_hub.csv"
OUTPUT_EXTRAS_PATH = "clean-data/waymo_nhtsa_crashes_not_in_hub.csv"

# Let's verify the files exist before we start
files_to_check = [
    ("NHTSA Prior (Amendment 2)", NHTSA_PRIOR_PATH),
    ("NHTSA Post (Amendment 3)", NHTSA_POST_PATH),
    ("Waymo Safety Hub CSV2", WAYMO_HUB_PATH),
]

print("Checking if files exist...")
print()
for name, path in files_to_check:
    if os.path.exists(path):
        print(f"✓ {name}: FOUND")
    else:
        print(f"✗ {name}: NOT FOUND at {path}")

Checking if files exist...

✗ NHTSA Prior (Amendment 2): NOT FOUND at raw-data/nhtsa_ads_prior_june16_downloaded20250129.csv
✗ NHTSA Post (Amendment 3): NOT FOUND at raw-data/nhtsa_ads_post_june16_downloaded_20250129.csv
✗ Waymo Safety Hub CSV2: NOT FOUND at raw-data/waymo_safety_hub_csv2_downloaded_20250129.csv


## Step 2: Load and Filter NHTSA Data (Prior to June 16)

The NHTSA data contains crashes from ALL AV companies (Waymo, Cruise, Zoox, Tesla, etc.).
We only want Waymo crashes, so we filter by `Reporting Entity`.

- `pd.read_csv()` reads the CSV into a DataFrame (like an Excel spreadsheet in Python)
- `.copy()` creates a fresh independent copy so we don't accidentally modify the original

In [6]:
# Load the NHTSA data from before June 16, 2025 (Amendment 2 format)
nhtsa_prior = pd.read_csv(NHTSA_PRIOR_PATH)

print("Total rows loaded:", len(nhtsa_prior))
print("Columns:", len(nhtsa_prior.columns))
print()

# See which companies are in this dataset
print("Reporting entities (top 10):")
print(nhtsa_prior['Reporting Entity'].value_counts().head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'raw-data/nhtsa_ads_prior_june16_downloaded20250129.csv'

In [ ]:
# Keep only rows where 'Reporting Entity' equals 'Waymo LLC'
waymo_prior = nhtsa_prior[nhtsa_prior['Reporting Entity'] == 'Waymo LLC'].copy()

print("Waymo crashes (prior to June 16):", len(waymo_prior))

Waymo crashes (prior to June 16): 1281


## Step 3: Load and Filter NHTSA Data (After June 16)

Same thing for the Amendment 3 data. This dataset has a different column structure
due to NHTSA's reporting changes, but the loading and filtering process is identical.

In [ ]:
# Load the NHTSA data from after June 16, 2025 (Amendment 3 format)
nhtsa_post = pd.read_csv(NHTSA_POST_PATH)

print("Total rows loaded:", len(nhtsa_post))
print("Columns:", len(nhtsa_post.columns))
print()

# See which companies are in this dataset
print("Reporting entities:")
print(nhtsa_post['Reporting Entity'].value_counts())

Total rows loaded: 481
Columns: 116

Reporting entities:
Reporting Entity
Waymo LLC                  419
Zoox, Inc.                  22
Tesla, Inc.                 10
May Mobility                 8
Motional                     5
Avride Inc.                  3
Aurora Operations, Inc.      3
Beep, Inc.                   3
Nuro                         3
Hyundai Motor America        2
Ohmio, Inc.                  1
Stack AV                     1
Oxbotica                     1
Name: count, dtype: int64


In [ ]:
# Keep only Waymo crashes from the post-June-16 data
waymo_post = nhtsa_post[nhtsa_post['Reporting Entity'] == 'Waymo LLC'].copy()

print("Waymo crashes (after June 16):", len(waymo_post))

Waymo crashes (after June 16): 419


## Step 4: Harmonize Column Names Between Amendments

Amendment 3 changed NHTSA's reporting format. Some columns contain the same
information but have different names. We rename the **prior** data to match the
**post** naming convention so we can stack them together.

### Columns we rename:
| Amendment 2 (Prior) | Amendment 3 (Post) | What changed |
|---|---|---|
| `Weather - Fog/Smoke` | `Weather - Fog/Smoke/Haze` | Added "Haze" |
| `SV Were All Passengers Belted?` | `Were All Passengers Belted?` | Removed "SV" prefix |
| `SV Any Air Bags Deployed?` | `Any Air Bags Deployed?` | Combined SV+CP into one field |
| `SV Was Vehicle Towed?` | `Was Any Vehicle Towed?` | Combined SV+CP into one field |

### Columns that only exist in ONE time period (limitations):
- **Prior only:** Posted Speed Limit, Lighting, Property Damage?, CP Air Bags, CP Towed
- **Post only:** Weather - Dust Storm, Partly Cloudy, Severe Hurricane, Structure-Indoor

In [ ]:
# Rename columns in the PRIOR dataset to match the POST naming convention
# We rename prior → post because Amendment 3 is the current/going-forward format

waymo_prior = waymo_prior.rename(columns={
    'Weather - Fog/Smoke': 'Weather - Fog/Smoke/Haze',
    'SV Were All Passengers Belted?': 'Were All Passengers Belted?',
    'SV Was Vehicle Towed?': 'Was Any Vehicle Towed?',
    'SV Any Air Bags Deployed?': 'Any Air Bags Deployed?',
})

print("✓ Renamed columns in prior dataset to match post dataset")

# Quick verification that the renames worked
print()
for col in ['Weather - Fog/Smoke/Haze', 'Were All Passengers Belted?', 'Was Any Vehicle Towed?', 'Any Air Bags Deployed?']:
    print(f"  '{col}' exists in prior: {col in waymo_prior.columns}")

✓ Renamed columns in prior dataset to match post dataset

  'Weather - Fog/Smoke/Haze' exists in prior: True
  'Were All Passengers Belted?' exists in prior: True
  'Was Any Vehicle Towed?' exists in prior: True
  'Any Air Bags Deployed?' exists in prior: True


## Step 5: Add Data Period Tracker and Combine NHTSA Datasets

Before combining, we add a `Data Period` column so we always know which amendment
each crash came from. Then we stack the two datasets on top of each other.

- `pd.concat()` stacks dataframes vertically (like stacking two Excel sheets)
- `ignore_index=True` gives us a fresh row index (0, 1, 2, ...) to avoid duplicates

In [ ]:
# Add a column to track which amendment period each crash came from
waymo_prior['Data Period'] = 'Prior to June 16, 2025 (Amendment 2)'
waymo_post['Data Period'] = 'After June 16, 2025 (Amendment 3)'

# Combine the two NHTSA datasets by stacking them on top of each other
# pd.concat() stacks them vertically; ignore_index=True gives a fresh row index
waymo_nhtsa_combined = pd.concat([waymo_prior, waymo_post], ignore_index=True)

print("Dataset sizes:")
print(f"  Prior to June 16:  {len(waymo_prior)} crashes")
print(f"  After June 16:     {len(waymo_post)} crashes")
print(f"  Combined total:    {len(waymo_nhtsa_combined)} crashes")
print()
print(f"Total columns in combined dataset: {len(waymo_nhtsa_combined.columns)}")
print()

# Show columns that only exist in one period (they'll have NaN in the other)
prior_only_cols = set(waymo_prior.columns) - set(waymo_post.columns)
post_only_cols = set(waymo_post.columns) - set(waymo_prior.columns)
print(f"Columns only in PRIOR (will be NaN for post-June crashes): {len(prior_only_cols)}")
for col in sorted(prior_only_cols):
    print(f"  - {col}")
print()
print(f"Columns only in POST (will be NaN for pre-June crashes): {len(post_only_cols)}")
for col in sorted(post_only_cols):
    print(f"  - {col}")

Dataset sizes:
  Prior to June 16:  1281 crashes
  After June 16:     419 crashes
  Combined total:    1700 crashes

Total columns in combined dataset: 161

Columns only in PRIOR (will be NaN for post-June crashes): 44
  - ADAS/ADS Hardware Version
  - ADAS/ADS Hardware Version - Unk
  - ADAS/ADS Hardware Version CBI
  - ADAS/ADS Software Version
  - ADAS/ADS Software Version - Unk
  - ADAS/ADS Software Version CBI
  - ADAS/ADS System Version
  - ADAS/ADS System Version - Unk
  - ADAS/ADS System Version CBI
  - ADS Equipped?
  - CP Any Air Bags Deployed?
  - CP Was Vehicle Towed?
  - Federal Reg. Exemption - No
  - Federal Reg. Exemption - Unk
  - Federal Regulatory Exemption?
  - Inv. Officer Email - Unknown
  - Inv. Officer Name - Unknown
  - Inv. Officer Phone - Unknown
  - Investigating Officer Email
  - Investigating Officer Name
  - Investigating Officer Phone
  - Law Enforcement Investigating?
  - Lighting
  - Mileage
  - Mileage - Unknown
  - Notice Received Date
  - Other Fede

## Step 6: Keep Only Latest Version of Each Crash

Some crashes have multiple report versions (updates submitted to NHTSA over time).
We want only the most recent version to avoid counting the same incident twice.

- `Report ID` identifies the crash (e.g., 30270-11853)
- `Report Version` tells us which update (1, 2, 3, etc.)
- We sort by version and keep the last (= highest) one per crash

In [ ]:
# First, let's see how many duplicates we have
total_rows = len(waymo_nhtsa_combined)
unique_ids = waymo_nhtsa_combined['Report ID'].nunique()
print(f"Total rows: {total_rows}")
print(f"Unique Report IDs: {unique_ids}")
print(f"Duplicate versions to remove: {total_rows - unique_ids}")

Total rows: 1700
Unique Report IDs: 1531
Duplicate versions to remove: 169


In [ ]:
# STEP 1: Sort by Report ID, then by Report Version (ascending)
# This puts all versions of the same crash together, with the highest version last
waymo_nhtsa_combined = waymo_nhtsa_combined.sort_values(['Report ID', 'Report Version'])

# STEP 2: Drop older versions, keep only the newest
# drop_duplicates() removes rows with the same Report ID
# keep='last' means: when duplicates exist, keep the last one (highest version)
waymo_nhtsa_combined = waymo_nhtsa_combined.drop_duplicates(subset='Report ID', keep='last')

# STEP 3: Verify — row count should now equal unique Report IDs
print(f"After deduplication: {len(waymo_nhtsa_combined)} rows")
print(f"Unique Report IDs:   {waymo_nhtsa_combined['Report ID'].nunique()}")
print()
print("✓ Each crash now has exactly one row (the latest version)")

After deduplication: 1531 rows
Unique Report IDs:   1531

✓ Each crash now has exactly one row (the latest version)


## Step 7: Load Waymo Safety Hub Data

Now we load Waymo's own curated dataset (CSV2 from the Safety Impact Data Hub).
This contains Waymo's own crash classifications like Crash Type, injury flags, etc.

The merge key:
- Waymo hub uses `SGO Report ID`
- NHTSA uses `Report ID`
- They contain the same values (e.g., "30270-11853")

In [ ]:
# Load the Waymo Safety Hub data
waymo_hub = pd.read_csv(WAYMO_HUB_PATH)

print("Waymo Safety Hub rows:", len(waymo_hub))
print("Columns:", len(waymo_hub.columns))
print()
print("Columns in the hub:")
for col in waymo_hub.columns:
    print(f"  - {col}")
print()
print(f"Year-Month range: {waymo_hub['Year Month'].min()} to {waymo_hub['Year Month'].max()}")

Waymo Safety Hub rows: 1123
Columns: 16

Columns in the hub:
  - SGO Report ID
  - SGO Report Version
  - SGO Amendment
  - Year Month
  - Location
  - Crash Type
  - Is NHTSA Reportable In-Transport
  - Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH
  - Is Police-Reported
  - Is Any-Injury-Reported
  - Is Any Vehicle Airbag Deployment
  - Is Ego Vehicle Airbag Deployment
  - Is Suspected Serious Injury+
  - Incident Date
  - Location Address / Description
  - Zip Code

Year-Month range: 202009 to 202509


## Step 8: Check Merge Key Compatibility

Before merging, let's verify how well the IDs match between the datasets.
This tells us what to expect in the merge output.

In [ ]:
# Get the unique IDs from each dataset
hub_ids = set(waymo_hub['SGO Report ID'].dropna())  # dropna() because some hub rows have no SGO ID
nhtsa_ids = set(waymo_nhtsa_combined['Report ID'].dropna())

# Calculate overlaps
in_both = hub_ids & nhtsa_ids                # Crashes in BOTH datasets (these will merge)
hub_only = hub_ids - nhtsa_ids                # In hub but not in NHTSA (will have empty NHTSA columns)
nhtsa_only = nhtsa_ids - hub_ids              # In NHTSA but not in hub (the "extras")

# Also count hub rows with no SGO Report ID at all
hub_no_id = waymo_hub['SGO Report ID'].isna().sum()

print("=== MERGE KEY DIAGNOSTICS ===")
print()
print(f"Waymo Hub unique IDs:       {len(hub_ids)}")
print(f"NHTSA combined unique IDs:  {len(nhtsa_ids)}")
print()
print(f"✓ Crashes in BOTH (will merge):           {len(in_both)}")
print(f"  Hub rows with no SGO Report ID:          {hub_no_id} (will stay, with empty NHTSA columns)")
print(f"  NHTSA crashes NOT in hub (extras):       {len(nhtsa_only)} (saved to separate file)")

=== MERGE KEY DIAGNOSTICS ===

Waymo Hub unique IDs:       1119
NHTSA combined unique IDs:  1531

✓ Crashes in BOTH (will merge):           1119
  Hub rows with no SGO Report ID:          2 (will stay, with empty NHTSA columns)
  NHTSA crashes NOT in hub (extras):       412 (saved to separate file)


## Step 9: Separate the "Extra" NHTSA Crashes

These are Waymo crashes in the NHTSA data that don't appear in the Safety Hub.
There are two reasons for this:

1. **Newer crashes** (Oct–Dec 2025): The Waymo hub updates quarterly, so these
   crashes happened after the hub's last update (data through September 2025).
2. **Excluded by Waymo**: Older crashes that don't meet the hub's inclusion criteria
   (e.g., not Rider-Only, Mountain View test crashes, etc.).

We save these to a separate CSV with a flag explaining why they're not in the hub.

In [ ]:
# Pull out the NHTSA crashes that are NOT in the Waymo hub
extras = waymo_nhtsa_combined[~waymo_nhtsa_combined['Report ID'].isin(hub_ids)].copy()

print(f"Total 'extra' NHTSA crashes not in hub: {len(extras)}")
print()

# Let's see the date distribution to understand why they're not in the hub
# The Waymo hub's last data point is September 2025 (YearMonth = 202509)
# So crashes from October 2025+ are simply "not yet updated"
print("Date distribution of extras:")
print(extras['Incident Date'].value_counts().head(15))
print()
print("By Data Period:")
print(extras['Data Period'].value_counts())

Total 'extra' NHTSA crashes not in hub: 412

Date distribution of extras:
Incident Date
NOV-2025    100
OCT-2025     86
NOV-2021     14
DEC-2025     13
JAN-2022     10
FEB-2022     10
APR-2022      9
AUG-2022      9
JUL-2022      8
OCT-2021      7
MAR-2022      7
MAY-2022      7
AUG-2021      6
SEP-2021      6
DEC-2021      6
Name: count, dtype: int64

By Data Period:
Data Period
After June 16, 2025 (Amendment 3)       208
Prior to June 16, 2025 (Amendment 2)    204
Name: count, dtype: int64


In [ ]:
# Add a flag explaining why each crash is not in the hub
# We use the Data Period to determine the reason:
# - Post-June crashes are likely newer than the hub's update cycle
# - Prior-June crashes were likely excluded by Waymo's methodology

extras['Reason Not In Hub'] = extras['Data Period'].apply(
    lambda x: 'Likely newer than hub update cycle (post-June 2025)' 
    if x == 'After June 16, 2025 (Amendment 3)' 
    else 'Likely excluded by Waymo hub methodology (pre-June 2025)'
)

print("Reason breakdown:")
print(extras['Reason Not In Hub'].value_counts())

# Save the extras to a separate CSV
extras.to_csv(OUTPUT_EXTRAS_PATH, index=False)
print()
print(f"✓ Saved {len(extras)} extra crashes to: {OUTPUT_EXTRAS_PATH}")

Reason breakdown:
Reason Not In Hub
Likely newer than hub update cycle (post-June 2025)         208
Likely excluded by Waymo hub methodology (pre-June 2025)    204
Name: count, dtype: int64

✓ Saved 412 extra crashes to: waymo_nhtsa_crashes_not_in_hub.csv


## Step 10: Merge Waymo Hub with NHTSA Data

Now the main event! We do a **left join** using the Waymo hub as the base.

- **Left join** means: keep ALL rows from the Waymo hub, and add NHTSA columns where there's a match
- Merge key: `SGO Report ID` (hub) = `Report ID` (NHTSA)
- Hub rows without a matching NHTSA record will have NaN in the NHTSA columns

We add the suffix `_nhtsa` to any NHTSA columns that have the same name as hub columns,
to avoid confusion (e.g., both have `Incident Date` and `Zip Code`).

In [ ]:
# First, remove the extras from the NHTSA data (they don't belong in the merged output)
# We only want NHTSA crashes that actually appear in the Waymo hub
nhtsa_for_merge = waymo_nhtsa_combined[waymo_nhtsa_combined['Report ID'].isin(hub_ids)].copy()

print(f"NHTSA rows going into merge: {len(nhtsa_for_merge)}")
print(f"Waymo hub rows: {len(waymo_hub)}")

NHTSA rows going into merge: 1119
Waymo hub rows: 1123


In [ ]:
# Perform the left join
# left_on / right_on tells pandas which columns to match on
# suffixes: if both datasets have a column with the same name (e.g., 'Incident Date'),
#   the hub version keeps its name, and the NHTSA version gets '_nhtsa' appended

merged = waymo_hub.merge(
    nhtsa_for_merge,
    left_on='SGO Report ID',
    right_on='Report ID',
    how='left',
    suffixes=('', '_nhtsa')
)

print(f"Merged dataset: {len(merged)} rows x {len(merged.columns)} columns")
print()

# Sanity check: merged rows should equal hub rows (since it's a left join)
if len(merged) == len(waymo_hub):
    print("✓ Row count matches the Waymo hub — no unexpected duplicates from the merge")
else:
    print(f"⚠ WARNING: Row count changed! Hub had {len(waymo_hub)}, merged has {len(merged)}")
    print("  This likely means some Report IDs in NHTSA have duplicates. Investigate!")

Merged dataset: 1123 rows x 177 columns

✓ Row count matches the Waymo hub — no unexpected duplicates from the merge


In [ ]:
# Let's see how many hub rows successfully matched to NHTSA data
matched = merged['Report ID'].notna().sum()
unmatched = merged['Report ID'].isna().sum()

print(f"Hub rows with NHTSA match:    {matched}")
print(f"Hub rows without NHTSA match: {unmatched} (these have NaN in NHTSA columns)")
print()

# Show the unmatched rows if any
if unmatched > 0:
    print("Unmatched hub rows (no SGO Report ID or no NHTSA record):")
    unmatched_rows = merged[merged['Report ID'].isna()]
    print(unmatched_rows[['SGO Report ID', 'Year Month', 'Location', 'Crash Type']].to_string())

Hub rows with NHTSA match:    1121
Hub rows without NHTSA match: 2 (these have NaN in NHTSA columns)

Unmatched hub rows (no SGO Report ID or no NHTSA record):
     SGO Report ID  Year Month Location Crash Type
1121           NaN      202105  PHOENIX    V2V F2R
1122           NaN      202009  PHOENIX    V2V F2R


## Step 11: Review the Merged Dataset

In [ ]:
# Show all columns in the merged dataset, grouped by source
hub_cols = list(waymo_hub.columns)
nhtsa_new_cols = [col for col in merged.columns if col not in hub_cols]

print(f"=== COLUMNS FROM WAYMO HUB ({len(hub_cols)}) ===")
for col in hub_cols:
    print(f"  {col}")

print()
print(f"=== NEW COLUMNS FROM NHTSA ({len(nhtsa_new_cols)}) ===")
for col in nhtsa_new_cols:
    print(f"  {col}")

=== COLUMNS FROM WAYMO HUB (16) ===
  SGO Report ID
  SGO Report Version
  SGO Amendment
  Year Month
  Location
  Crash Type
  Is NHTSA Reportable In-Transport
  Is NHTSA Reportable In-Transport Delta-V Less than 1 MPH
  Is Police-Reported
  Is Any-Injury-Reported
  Is Any Vehicle Airbag Deployment
  Is Ego Vehicle Airbag Deployment
  Is Suspected Serious Injury+
  Incident Date
  Location Address / Description
  Zip Code

=== NEW COLUMNS FROM NHTSA (161) ===
  Report ID
  Report Version
  Reporting Entity
  Report Type
  Report Month
  Report Year
  Report Submission Date
  VIN
  VIN - Unknown
  Serial Number
  Make
  Model
  Model - Unknown
  Model Year
  Model Year - Unknown
  Same Vehicle ID
  Mileage
  Mileage - Unknown
  Driver / Operator Type
  ADAS/ADS System Version
  ADAS/ADS System Version - Unk
  ADAS/ADS System Version CBI
  ADAS/ADS Hardware Version
  ADAS/ADS Hardware Version - Unk
  ADAS/ADS Hardware Version CBI
  ADAS/ADS Software Version
  ADAS/ADS Software Version -

In [ ]:
# Quick look at a few key NHTSA columns to make sure the merge worked
print("=== SAMPLE OF KEY MERGED COLUMNS ===")
print()

sample_cols = ['SGO Report ID', 'Crash Type', 'Location', 
               'Crash With', 'SV Pre-Crash Movement', 'Highest Injury Severity Alleged',
               'Narrative']

# Only show columns that exist (some might not if merge had issues)
existing_sample_cols = [col for col in sample_cols if col in merged.columns]
print(merged[existing_sample_cols].head(5).to_string())

=== SAMPLE OF KEY MERGED COLUMNS ===

  SGO Report ID   Crash Type       Location     Crash With SV Pre-Crash Movement       Highest Injury Severity Alleged                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

## Step 12: Save the Merged Dataset

In [ ]:
# Save the final merged dataset
merged.to_csv(OUTPUT_MERGED_PATH, index=False)

print(f"✓ Merged dataset saved to: {OUTPUT_MERGED_PATH}")
print(f"  Rows: {len(merged)}")
print(f"  Columns: {len(merged.columns)}")
print()
print(f"✓ Extra NHTSA crashes saved to: {OUTPUT_EXTRAS_PATH}")
print(f"  Rows: {len(extras)}")

✓ Merged dataset saved to: waymo_merged_nhtsa_hub.csv
  Rows: 1123
  Columns: 177

✓ Extra NHTSA crashes saved to: waymo_nhtsa_crashes_not_in_hub.csv
  Rows: 412


## Step 13: Final Summary

A complete overview of what happened during this merge.

In [ ]:
print("="*60)
print("MERGE SUMMARY")
print("="*60)
print()
print("INPUT FILES:")
print(f"  NHTSA Prior (Amendment 2):  {len(waymo_prior)} Waymo crashes")
print(f"  NHTSA Post (Amendment 3):   {len(waymo_post)} Waymo crashes")
print(f"  Waymo Safety Hub CSV2:      {len(waymo_hub)} crashes")
print()
print("PROCESSING:")
print(f"  Combined NHTSA (after dedup): {len(waymo_nhtsa_combined)} unique crashes")
print(f"  Successfully matched to hub:  {matched}")
print(f"  Hub rows without NHTSA match: {unmatched}")
print()
print("OUTPUT FILES:")
print(f"  1. {OUTPUT_MERGED_PATH}")
print(f"     → {len(merged)} rows x {len(merged.columns)} columns")
print(f"     → Waymo hub enriched with ALL NHTSA columns")
print()
print(f"  2. {OUTPUT_EXTRAS_PATH}")
print(f"     → {len(extras)} crashes in NHTSA but not in Waymo hub")
print(f"     → Reason breakdown:")
for reason, count in extras['Reason Not In Hub'].value_counts().items():
    print(f"        {count} — {reason}")
print()
print("COLUMN NOTES:")
print("  - Columns with '_nhtsa' suffix: duplicated between hub and NHTSA")
print("  - Prior-only columns (e.g., Posted Speed Limit): NaN for post-June crashes")
print("  - Post-only columns (e.g., Weather - Dust Storm): NaN for pre-June crashes")

MERGE SUMMARY

INPUT FILES:
  NHTSA Prior (Amendment 2):  1281 Waymo crashes
  NHTSA Post (Amendment 3):   419 Waymo crashes
  Waymo Safety Hub CSV2:      1123 crashes

PROCESSING:
  Combined NHTSA (after dedup): 1531 unique crashes
  Successfully matched to hub:  1121
  Hub rows without NHTSA match: 2

OUTPUT FILES:
  1. waymo_merged_nhtsa_hub.csv
     → 1123 rows x 177 columns
     → Waymo hub enriched with ALL NHTSA columns

  2. waymo_nhtsa_crashes_not_in_hub.csv
     → 412 crashes in NHTSA but not in Waymo hub
     → Reason breakdown:
        208 — Likely newer than hub update cycle (post-June 2025)
        204 — Likely excluded by Waymo hub methodology (pre-June 2025)

COLUMN NOTES:
  - Columns with '_nhtsa' suffix: duplicated between hub and NHTSA
  - Prior-only columns (e.g., Posted Speed Limit): NaN for post-June crashes
  - Post-only columns (e.g., Weather - Dust Storm): NaN for pre-June crashes
